# 04a — CNN-LSTM Final: ECG Forecasting (4.9s → 4.9s)

### Key changes from previous version

1. **SpikeWeightedMSELoss** — spikes get 5x more penalty than baseline
2. **LSTM-CNN architecture** — no pooling, full resolution preserved
3. **Deeper model** — 9 dilated blocks, 96 channels, ~1.2M params
4. **Cosine annealing** with warm restarts for better convergence
5. **Longer training** — 120 epochs, patience 20

In [4]:
# CELL 1 — IMPORTS
import os, pickle, warnings, math
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR, ReduceLROnPlateau
from sklearn.metrics import mean_absolute_error, mean_squared_error

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch : {torch.__version__}")
print(f"Device  : {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
print("✅ Imports ready")

PyTorch : 2.11.0
Device  : cpu
✅ Imports ready


In [5]:
# CELL 2 — LOAD DATA
SAVE_DIR = os.path.join('..', 'data', 'processed')
FIG_DIR  = os.path.join('..', 'reports', 'figures', 'cnn_lstm')
CKPT_DIR = os.path.join('..', 'reports', 'checkpoints')
os.makedirs(FIG_DIR,  exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

X_train = np.load(os.path.join(SAVE_DIR, 'X_train.npy'))
y_train = np.load(os.path.join(SAVE_DIR, 'y_train.npy'))
X_val   = np.load(os.path.join(SAVE_DIR, 'X_val.npy'))
y_val   = np.load(os.path.join(SAVE_DIR, 'y_val.npy'))
X_test  = np.load(os.path.join(SAVE_DIR, 'X_test.npy'))
y_test  = np.load(os.path.join(SAVE_DIR, 'y_test.npy'))

with open(os.path.join(SAVE_DIR, 'config.pkl'), 'rb') as f:
    cfg = pickle.load(f)

LEAD_NAMES = cfg['lead_names']
FS         = cfg['sampling_rate']
INPUT_LEN  = cfg['input_len']
HORIZON    = cfg['horizon']
N_LEADS    = cfg['n_leads']

print(f"X_train : {X_train.shape}  y_train : {y_train.shape}")
print(f"X_val   : {X_val.shape}    y_val   : {y_val.shape}")
print(f"X_test  : {X_test.shape}   y_test  : {y_test.shape}")
print(f"Input   : {INPUT_LEN} samples = {INPUT_LEN/FS:.2f}s")
print(f"Horizon : {HORIZON}  samples = {HORIZON/FS:.2f}s")
print("✅ Data loaded")

X_train : (31362, 490, 12)  y_train : (31362, 490, 12)
X_val   : (3920, 490, 12)    y_val   : (3920, 490, 12)
X_test  : (2198, 490, 12)   y_test  : (2198, 490, 12)
Input   : 490 samples = 4.90s
Horizon : 490  samples = 4.90s
✅ Data loaded


In [6]:
# CELL 3 — DATALOADERS
class ECGForecastDataset(Dataset):
    def __init__(self, X, y, channel_first=False):
        self.X = torch.from_numpy(np.asarray(X, dtype=np.float32))
        self.y = torch.from_numpy(np.asarray(y, dtype=np.float32))
        self.channel_first = channel_first
    def __len__(self): return len(self.X)
    def __getitem__(self, idx):
        x, y = self.X[idx], self.y[idx]
        if self.channel_first: x = x.permute(1, 0)
        return x, y

def make_loaders(X_tr, y_tr, X_v, y_v, X_te, y_te,
                 channel_first=False, batch_train=64, batch_eval=128):
    kw = dict(num_workers=0, pin_memory=(DEVICE.type == 'cuda'))
    tr = DataLoader(ECGForecastDataset(X_tr, y_tr, channel_first),
                    batch_size=batch_train, shuffle=True, drop_last=True, **kw)
    vl = DataLoader(ECGForecastDataset(X_v,  y_v,  channel_first),
                    batch_size=batch_eval,  shuffle=False, **kw)
    te = DataLoader(ECGForecastDataset(X_te, y_te, channel_first),
                    batch_size=batch_eval,  shuffle=False, **kw)
    return tr, vl, te

cnn_tr, cnn_vl, cnn_te = make_loaders(
    X_train, y_train, X_val, y_val, X_test, y_test, channel_first=True)

xb, yb = next(iter(cnn_tr))
print(f"Batch : x={xb.shape}  y={yb.shape}")
print(f"Batches : train={len(cnn_tr)} | val={len(cnn_vl)} | test={len(cnn_te)}")
print("✅ DataLoaders ready")

Batch : x=torch.Size([64, 12, 490])  y=torch.Size([64, 490, 12])
Batches : train=490 | val=31 | test=18
✅ DataLoaders ready


In [7]:
# CELL 4 — MODEL: LSTM-CNN Dilated Conv (NO POOLING)
#
# Key design:
#   - No pooling = no spike information lost
#   - Dilated convolutions grow receptive field exponentially
#   - 9 blocks with dilations [1,2,4,8,16,32,64,128,256] = RF 1197 steps
#   - 96 channels for more capacity
#   - Direct per-timestep output = spikes appear exactly where they should

class DilatedBlock(nn.Module):
    def __init__(self, channels, dilation, kernel_size=7, dropout=0.15):
        super().__init__()
        pad = (kernel_size - 1) * dilation // 2
        self.conv1 = nn.Conv1d(channels, channels, kernel_size,
                               dilation=dilation, padding=pad, bias=False)
        self.conv2 = nn.Conv1d(channels, channels, kernel_size,
                               dilation=dilation, padding=pad, bias=False)
        self.bn1   = nn.BatchNorm1d(channels)
        self.bn2   = nn.BatchNorm1d(channels)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x):
        res = x
        out = F.gelu(self.bn1(self.conv1(x)))
        out = self.drop(out)
        out = F.gelu(self.bn2(self.conv2(out)))
        out = self.drop(out)
        return out + res


class CNNLSTMForecaster(nn.Module):
    DILATIONS   = [1, 2, 4, 8, 16, 32, 64, 128, 256]
    CHANNELS    = 96
    KERNEL_SIZE = 7

    def __init__(self, n_leads=12, horizon=490, dropout=0.15):
        super().__init__()
        self.horizon = horizon
        self.n_leads = n_leads

        self.input_proj = nn.Sequential(
            nn.Conv1d(n_leads, self.CHANNELS, kernel_size=1, bias=False),
            nn.BatchNorm1d(self.CHANNELS),
            nn.GELU())

        self.blocks = nn.ModuleList([
            DilatedBlock(self.CHANNELS, d, self.KERNEL_SIZE, dropout)
            for d in self.DILATIONS
        ])

        self.output_proj = nn.Sequential(
            nn.Conv1d(self.CHANNELS, self.CHANNELS, kernel_size=1, bias=False),
            nn.BatchNorm1d(self.CHANNELS),
            nn.GELU(),
            nn.Conv1d(self.CHANNELS, n_leads, kernel_size=1))

    def forward(self, x):
        out = self.input_proj(x)
        for block in self.blocks:
            out = block(out)
        out = self.output_proj(out)
        return out.permute(0, 2, 1)

    def enable_mc_dropout(self):
        self.eval()
        for m in self.modules():
            if isinstance(m, nn.Dropout): m.train()

rf = CNNLSTMForecaster.KERNEL_SIZE * sum(CNNLSTMForecaster.DILATIONS)
print(f"Receptive field : {rf} steps = {rf/100:.2f}s  (input={INPUT_LEN/FS:.2f}s)")
print(f"Covers full input: {rf >= INPUT_LEN}")
print("✅ CNNLSTMForecaster defined — NO POOLING, full spike resolution")

Receptive field : 3577 steps = 35.77s  (input=4.90s)
Covers full input: True
✅ CNNLSTMForecaster defined — NO POOLING, full spike resolution


In [8]:
# CELL 5 — INSTANTIATE & VERIFY
model    = CNNLSTMForecaster(
    n_leads=N_LEADS, horizon=HORIZON, dropout=0.15).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

with torch.no_grad():
    dummy = torch.randn(4, N_LEADS, INPUT_LEN).to(DEVICE)
    out   = model(dummy)
    print(f"✅ Forward pass: {dummy.shape} → {out.shape}")
    assert out.shape == (4, HORIZON, N_LEADS), f"Shape mismatch: {out.shape}"

print(f"Parameters : {n_params:,}")
print(f"Device     : {DEVICE}")
print("✅ Model ready")

✅ Forward pass: torch.Size([4, 12, 490]) → torch.Size([4, 490, 12])
Parameters : 1,176,588
Device     : cpu
✅ Model ready


In [9]:
# CELL 6 — SPIKE-WEIGHTED MSE LOSS
#
# Why this matters:
#   Plain MSE treats all timesteps equally. Spikes are ~5-10% of the signal
#   but carry all the clinical information. The model can get low MSE by
#   just predicting the mean (flat line) — spikes barely affect total loss.
#
# SpikeWeightedMSELoss:
#   1. Computes per-sample weight: abs(target) > 1.5*std → weight = spike_w
#   2. Spike regions get 5x more penalty than baseline
#   3. Model is FORCED to learn spikes or pay a huge loss penalty

class SpikeWeightedMSELoss(nn.Module):
    def __init__(self, spike_weight=5.0, spike_threshold=1.5):
        super().__init__()
        self.spike_weight = spike_weight
        self.spike_threshold = spike_threshold

    def forward(self, pred, target):
        # Per-element MSE
        mse = (pred - target) ** 2
        # Weight: 1.0 for baseline, spike_weight for spikes
        std = target.std(dim=(1, 2), keepdim=True) + 1e-8
        spike_mask = (target.abs() > self.spike_threshold * std).float()
        weight = 1.0 + (self.spike_weight - 1.0) * spike_mask
        return (weight * mse).mean()

criterion = SpikeWeightedMSELoss(spike_weight=5.0, spike_threshold=1.5)
print("✅ SpikeWeightedMSELoss — baseline=1.0x, spike=5.0x penalty")

✅ SpikeWeightedMSELoss — baseline=1.0x, spike=5.0x penalty


In [10]:
# CELL 7 — TRAINING ENGINE
def train_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    for xb, yb in tqdm(loader, desc='Train', leave=False, ncols=90):
        xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * len(xb)
    return total_loss / len(loader.dataset)

@torch.no_grad()
def eval_epoch(model, loader, device):
    model.eval()
    total_loss, preds, targets = 0.0, [], []
    for xb, yb in loader:
        xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
        pred   = model(xb)
        loss   = criterion(pred, yb)
        total_loss += loss.item() * len(xb)
        preds.append(pred.cpu().numpy())
        targets.append(yb.cpu().numpy())
    return total_loss / len(loader.dataset), np.concatenate(preds), np.concatenate(targets)


def train_model(model, tr_loader, vl_loader, n_epochs=120, lr=3e-4, patience=20):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=n_epochs, eta_min=1e-6)

    best_val, no_improve = float('inf'), 0
    ckpt    = os.path.join(CKPT_DIR, 'CNN-LSTM_final.pt')
    history = {'train_loss': [], 'val_loss': [], 'lr': []}

    print(f"\n{'─'*70}")
    print(f"  LSTM-CNN  |  {INPUT_LEN/FS:.1f}s→{HORIZON/FS:.1f}s  |  {n_params:,} params")
    print(f"  SpikeWeightedMSELoss | CosineAnnealing | batch=64")
    print(f"{'─'*70}")

    pbar = tqdm(range(1, n_epochs+1), desc='Epochs', unit='ep', ncols=90)
    for ep in pbar:
        tr_loss          = train_epoch(model, tr_loader, optimizer, DEVICE)
        vl_loss, _, _    = eval_epoch(model, vl_loader, DEVICE)
        current_lr       = optimizer.param_groups[0]['lr']
        scheduler.step()

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(vl_loss)
        history['lr'].append(current_lr)

        is_best = vl_loss < best_val
        if is_best:
            best_val = vl_loss; no_improve = 0
            torch.save(model.state_dict(), ckpt)
        else:
            no_improve += 1

        pbar.set_postfix({'tr': f'{tr_loss:.5f}', 'vl': f'{vl_loss:.5f}',
                          'lr': f'{current_lr:.1e}', 'pat': no_improve})
        if ep % 10 == 0 or is_best:
            tqdm.write(f"  {ep:3d} | train={tr_loss:.5f}  val={vl_loss:.5f}"
                       f"  lr={current_lr:.2e}"
                       f"{'  ★' if is_best else f'  ({no_improve}/{patience})'}")

        if no_improve >= patience:
            tqdm.write(f"  ⏹ Early stop ep {ep}  best={best_val:.6f}")
            break

    model.load_state_dict(torch.load(ckpt, map_location=DEVICE, weights_only=True))
    print(f"\n  Best val : {best_val:.6f}  saved → {ckpt}")
    print(f"{'─'*70}\n")
    return history

print("✅ Training engine ready")

✅ Training engine ready


In [ ]:
# CELL 8 — TRAIN
history = train_model(model, cnn_tr, cnn_vl, n_epochs=120, lr=3e-4, patience=20)


──────────────────────────────────────────────────────────────────────
  LSTM-CNN  |  4.9s→4.9s  |  1,176,588 params
  SpikeWeightedMSELoss | CosineAnnealing | batch=64
──────────────────────────────────────────────────────────────────────


Epochs:   0%|                                                     | 0/120 [00:00<?, ?ep/s]

Train:   0%|                                                      | 0/490 [00:00<?, ?it/s]

    1 | train=4.44508  val=4.10488  lr=3.00e-04  ★


Train:   0%|                                                      | 0/490 [00:00<?, ?it/s]

...

Train:   0%|                                                      | 0/490 [00:00<?, ?it/s]

    6 | train=3.61222  val=3.76342  lr=2.99e-04  ★


Train:   0%|                                                      | 0/490 [00:00<?, ?it/s]

    7 | train=3.55021  val=3.75209  lr=2.98e-04  ★


Train:   0%|                                                      | 0/490 [00:00<?, ?it/s]

In [ ]:
# CELL 9 — TRAINING HISTORY
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ep = range(1, len(history['train_loss'])+1)

axes[0].plot(ep, history['train_loss'], color='#0ea5e9', lw=2,
             label='Train', marker='o', ms=3)
axes[0].plot(ep, history['val_loss'],   color='#ef4444', lw=2,
             label='Val',   marker='s', ms=3, ls='--')
best_ep = int(np.argmin(history['val_loss'])) + 1
axes[0].axvline(best_ep, color='gold', ls=':', lw=2, label=f'Best ep {best_ep}')
axes[0].set_title(f'LSTM-CNN Loss ({INPUT_LEN/FS:.1f}s→{HORIZON/FS:.1f}s)',
                  fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('SpikeWeighted MSE')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(ep, history['lr'], color='#10b981', lw=2, marker='o', ms=3)
axes[1].set_title('LR Schedule (CosineAnnealing)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('LR')
axes[1].set_yscale('log'); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'training_history.png'), dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved training_history.png")

In [ ]:
# CELL 10 — TEST EVALUATION
test_loss, test_preds, test_targets = eval_epoch(model, cnn_te, DEVICE)

mae_per_lead, rmse_per_lead = [], []
for i in range(N_LEADS):
    p = test_preds[:, :, i].flatten()
    t = test_targets[:, :, i].flatten()
    mae_per_lead.append(mean_absolute_error(t, p))
    rmse_per_lead.append(np.sqrt(mean_squared_error(t, p)))

mae_macro  = np.mean(mae_per_lead)
rmse_macro = np.mean(rmse_per_lead)

print(f"Test SpikeWeightedMSE : {test_loss:.6f}")
print(f"\nPer-Lead Results:")
for i, name in enumerate(LEAD_NAMES):
    print(f"  {name:>4s}: MAE={mae_per_lead[i]:.4f}  RMSE={rmse_per_lead[i]:.4f}")
print(f"\nMacro MAE  : {mae_macro:.6f} mV")
print(f"Macro RMSE : {rmse_macro:.6f} mV")
print("✅ Evaluation complete")

In [ ]:
# CELL 11 — PER-LEAD RMSE
fig, ax = plt.subplots(figsize=(13, 6))
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, N_LEADS))
bars   = ax.bar(np.arange(N_LEADS), rmse_per_lead, width=0.6,
                color=colors, alpha=0.85, edgecolor='black', lw=0.5)
ax.axhline(rmse_macro, color='#facc15', ls='--', lw=2.5,
           label=f'Macro RMSE: {rmse_macro:.4f}')
for bar, val in zip(bars, rmse_per_lead):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height(),
            f'{val:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_xticks(np.arange(N_LEADS))
ax.set_xticklabels(LEAD_NAMES, fontsize=11, fontweight='bold')
ax.set_ylabel('RMSE (mV)', fontsize=12)
ax.set_title(f'LSTM-CNN Per-Lead RMSE ({INPUT_LEN/FS:.1f}s→{HORIZON/FS:.1f}s)',
             fontsize=14, fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'per_lead_rmse.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# CELL 12 — PREDICTED vs ACTUAL (spike check)
n_samples    = 4
lead_indices = [0, 1, 2, 6]   # I, II, III, V1
t = np.arange(HORIZON) / FS

fig, axes = plt.subplots(n_samples, 4, figsize=(20, 14))
for row in range(n_samples):
    for col, lead_idx in enumerate(lead_indices):
        ax     = axes[row, col]
        actual = test_targets[row, :, lead_idx]
        pred   = test_preds[row,   :, lead_idx]

        ax.plot(t, actual, color='#0ea5e9', lw=2,   label='Actual',    alpha=0.9)
        ax.plot(t, pred,   color='#ef4444', lw=1.5, label='Predicted', alpha=0.9, ls='--')

        rmse_i = np.sqrt(mean_squared_error(actual, pred))
        ax.set_title(f'{LEAD_NAMES[lead_idx]}  RMSE={rmse_i:.3f}',
                     fontsize=10, fontweight='bold')
        ax.set_xlabel('Time (s)'); ax.set_ylabel('mV')
        ax.grid(True, alpha=0.3)
        if col == 0: ax.legend(fontsize=8)

fig.suptitle(f'LSTM-CNN: Predicted vs Actual — {HORIZON/FS:.1f}s horizon',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'predictions_overlay.png'), dpi=150, bbox_inches='tight')
plt.show()
print("✅ Overlay saved")

In [ ]:
# CELL 13 — SPIKE TRACKING ANALYSIS
spike_mae, base_mae = [], []
for i in range(N_LEADS):
    actual = test_targets[:, :, i]
    pred   = test_preds[:,   :, i]
    thresh = actual.std() * 1.5
    sm = np.abs(actual) > thresh
    bm = ~sm
    if sm.sum() > 0: spike_mae.append(np.abs(actual[sm] - pred[sm]).mean())
    if bm.sum() > 0: base_mae.append( np.abs(actual[bm] - pred[bm]).mean())

spike_mean = np.mean(spike_mae)
base_mean  = np.mean(base_mae)
ratio      = spike_mean / base_mean

print(f"Baseline MAE (flat regions) : {base_mean:.4f} mV")
print(f"Spike MAE   (|sig| > 1.5σ)  : {spike_mean:.4f} mV")
print(f"Spike/base ratio            : {ratio:.2f}x")
if   ratio < 2.5: print("✅ Excellent spike tracking")
elif ratio < 4.0: print("✅ Good spike tracking")
elif ratio < 6.0: print("⚠  Partial spike tracking")
else:             print("❌ Spikes still missed")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Grouped bar chart
x = np.arange(N_LEADS)
bm_l, sm_l = [], []
for i in range(N_LEADS):
    a = test_targets[:,:,i]; p = test_preds[:,:,i]
    thr = a.std()*1.5
    sm  = np.abs(a) > thr; bm = ~sm
    sm_l.append(np.abs(a[sm]-p[sm]).mean() if sm.sum()>0 else 0)
    bm_l.append(np.abs(a[bm]-p[bm]).mean() if bm.sum()>0 else 0)

axes[0].bar(x-0.2, bm_l, 0.4, label='Baseline MAE',
            color='#0ea5e9', alpha=0.85, edgecolor='black')
axes[0].bar(x+0.2, sm_l, 0.4, label='Spike MAE',
            color='#ef4444', alpha=0.85, edgecolor='black')
axes[0].set_xticks(x); axes[0].set_xticklabels(LEAD_NAMES, fontsize=10, fontweight='bold')
axes[0].set_ylabel('MAE (mV)'); axes[0].legend()
axes[0].set_title('Baseline vs Spike MAE', fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='y')

# Scatter actual vs predicted (Lead II)
af = test_targets[:,:,1].flatten(); pf = test_preds[:,:,1].flatten()
idx = np.random.choice(len(af), 5000, replace=False)
axes[1].scatter(af[idx], pf[idx], alpha=0.15, s=4, color='#10b981')
mn = min(af.min(), pf.min()); mx = max(af.max(), pf.max())
axes[1].plot([mn,mx],[mn,mx],'r--',lw=2,label='Perfect')
axes[1].set_xlabel('Actual (mV)'); axes[1].set_ylabel('Predicted (mV)')
axes[1].set_title('Actual vs Predicted — Lead II', fontweight='bold')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'spike_tracking.png'), dpi=150, bbox_inches='tight')
plt.show()
print("✅ Spike analysis saved")

In [ ]:
# CELL 14 — FLAT-LINE CHECK
pred_std = test_preds.std(axis=0)
true_std = test_targets.std(axis=0)
t = np.arange(HORIZON) / FS

fig, axes = plt.subplots(3, 4, figsize=(18, 10))
for i, (ax, name) in enumerate(zip(axes.flatten(), LEAD_NAMES)):
    ax.plot(t, true_std[:, i], color='#0ea5e9', lw=1.5, label='Actual std')
    ax.plot(t, pred_std[:, i], color='#ef4444', lw=1.5, label='Pred std', ls='--')
    ax.set_title(f'Lead {name}', fontweight='bold', fontsize=10)
    ax.set_xlabel('Time (s)'); ax.set_ylabel('Std (mV)'); ax.grid(alpha=0.3)
    if i == 0: ax.legend(fontsize=8)
    if pred_std[:, i].min() < 0.01:
        ax.set_facecolor('#fff0f0')
        ax.set_title(f'Lead {name} ⚠ flat?', color='red', fontweight='bold', fontsize=10)

fig.suptitle('LSTM-CNN — Variance check (flat-line diagnostic)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'variance_check.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"Min pred std: {pred_std.min():.4f}  "
      f"{'✅ No flat lines' if pred_std.min() >= 0.01 else '⚠ flat?'}")

In [ ]:
# CELL 15 — SAVE RESULTS
results = {
    'model': 'LSTM-CNN_SpikeWeighted',
    'horizon_s': HORIZON/FS, 'input_s': INPUT_LEN/FS,
    'n_parameters': n_params, 'test_mse': test_loss,
    'mae_per_lead': mae_per_lead, 'mae_macro': mae_macro,
    'rmse_per_lead': rmse_per_lead, 'rmse_macro': rmse_macro,
    'history': history, 'lead_names': LEAD_NAMES,
    'test_preds': test_preds, 'test_targets': test_targets,
}
path = os.path.join(CKPT_DIR, 'CNN-LSTM_results_summary.pkl')
with open(path, 'wb') as f: pickle.dump(results, f)

print(f"{'='*60}")
print(f"  LSTM-CNN FINAL  ({INPUT_LEN/FS:.1f}s → {HORIZON/FS:.1f}s)")
print(f"{'='*60}")
print(f"  Params     : {n_params:,}")
print(f"  Test MSE   : {test_loss:.6f}")
print(f"  Macro MAE  : {mae_macro:.6f} mV")
print(f"  Macro RMSE : {rmse_macro:.6f} mV")
print(f"  Saved      → {path}")
print("✅ Done!")